# Изучение набора данных


Любой пространственный анализ начинается с предварительного изучения данных: необходимо понять структуру набора данных, его содержание и другие характеристики:

- какие объекты представлены в данных;
- какой тип геометрии используется (точки, линии, полигоны);
- в какой системе координат заданы данные;
- какие атрибуты доступны для анализа;
- присутствуют ли пропуски, дубликаты или потенциальные ошибки.

> В данном разделе мы рассмотрим, как выполнить первичное исследование набора пространственных данных в формате **GeoDataFrame**.


## 0. Импорт библиотек и подготовка данных


Импорт библиотек


In [ ]:
import geopandas as gpd

- [**GeoPandas**](https://geopandas.org/) (`geopandas`) — библиотека Python, расширение библиотеки pandas, предназначенное для работы с геопространственными данными. Позволяет загружать, обрабатывать и анализировать пространственные наборы данных в различных форматах.


Подготовка данных


Разберём всё на примере данных из нашего [репозитория](https://github.com/bella-mir/geoPython/tree/main/chapters/module_1/data):

- **spb_metro.geojson** — точки станций метро Санкт-Петербурга (2023).
  Источник: [Портал открытых данных Санкт-Петербурга](https://data.gov.spb.ru/)

При желании вы можете использовать собственные данные (это даже рекомендуется).

Сначала прочитаем файл в формате `.geojson`.


In [ ]:
gdf = gpd.read_file("data/spb_metro.geojson") 

## 1. Основная информация


### 1.1. Просмотр первых строк

Метод `head()` отображает первые строки `DataFrame` или `GeoDataFrame`.
Он используется для первичного знакомства с содержимым данных.

По умолчанию выводятся первые **5 строк**.


In [ ]:
gdf.head()

Можно указать количество строк вручную:


In [ ]:
gdf.head(3)

### 1.2. Общая информация

Метод `info()` отображает краткую сводную информацию о `DataFrame` или `GeoDataFrame`:

- количество записей (строк),
- названия полей,
- типы данных в каждом поле,
- количество **непустых значений**,
- объём памяти, занимаемый данными.


In [ ]:
gdf.info()

### 1.3. Размер

Чтобы понять размер набора данных, удобно использовать метод `len()` и атрибут `shape`.

Метод `len()` возвращает количество строк, то есть **количество объектов**.


In [ ]:
len(gdf)

Атрибут `shape` возвращает кортеж из двух чисел:

- первое число — количество строк (объектов),
- второе число — количество полей (атрибутов, включая `geometry`).


In [ ]:
gdf.shape

## 2. Геометрия


В `GeoDataFrame` каждый объект имеет **геометрию**

Перед выполнением пространственного анализа важно ответить на несколько вопросов:

- какой тип геометрии используется (точки, линии, полигоны);
- есть ли отсутствующие значения;
- корректны ли геометрии.


### 2.1. Формат геометрии

В `GeoDataFrame` пространственная информация хранится в специальном поле `geometry`.
Именно оно отличает `GeoDataFrame` от обычного `DataFrame`.

Этот атрибут возвращает объект типа `GeoSeries`, содержащий геометрию каждого пространственного объекта.

Посмотрим на первые пять значений:


In [ ]:
gdf.geometry[:5]

#### 2.1.1. Пропущенные значения

Иногда в наборе данных встречаются объекты без геометрии, что может приводить к некорректной работе инструментов пространственного анализа.
Поэтому важно заранее проверить их наличие и количество.

- `isna()` — определяет пропущенные значения;
- `sum()` — подсчитывает их количество.


In [ ]:
gdf.geometry.isna().sum()

Если значение больше нуля, значит часть строк содержит `NaN`, и такие объекты необходимо обработать (например, удалить или восстановить геометрию).

Удалить строки с пропущенной геометрией можно, например, следующим образом:


In [ ]:
gdf = gdf.dropna(subset=["geometry"])

#### 2.1.2. Пустые значения

Помимо пропущенных значений (`NaN`), в наборе данных могут встречаться **пустые геометрии** (`EMPTY`).
Формально геометрия у таких объектов существует, но она не содержит пространственной информации.

- `is_empty()` — определяет пустые геометрии;
- `sum()` — подсчитывает их количество.


In [ ]:
gdf.geometry.is_empty.sum()

Если значение больше нуля, значит в наборе данных есть пустые геометрии, и такие объекты необходимо обработать (например, удалить или восстановить геометрию).

Удалить строки с пропущенной геометрией можно, например, следующим образом:


In [ ]:
gdf = gdf[~gdf.geometry.is_empty]

#### 2.1.3. Активное поле геометрии

В `GeoDataFrame` всегда существует **активное поле геометрии** — именно оно используется для всех пространственных операций.

По умолчанию имя этого поля — `geometry`, однако оно может отличаться (например, после переименования столбца или создания новой геометрии).

Атрибут `geometry.name` позволяет узнать имя текущего активного геометрического поля.


In [ ]:
gdf.geometry.name

### 2.2. Тип геометрии

Чтобы определить тип геометрии объектов, используется атрибут `geom_type`.
Он возвращает объект типа `GeoSeries`, в котором для каждой строки указан тип геометрии.

Выведем первые пять значений:


In [ ]:
gdf.geom_type[:5]

Иногда в наборе данных могут встречаться объекты с разными типами геометрии.
Посмотрим, какие типы присутствуют в данных, и подсчитаем их количество.


In [ ]:
gdf.geom_type.value_counts()

### 2.3. Корректность геометрии

Помимо пропусков и пустых объектов, геометрия может быть **геометрически некорректной**.

Чаще всего проблемы с некорректной геометрией возникают у полигонов, например:

- самопересечения (self-intersection);
- незамкнутые контуры;
- дублирующиеся вершины;
- некорректная топология.

Атрибут `is_valid` позволяет проверить корректность геометрии каждого объекта. Он возвращает `True` или `False` для каждой геометрии.
Посчитаем количество некорректных геометрий в данных.


In [ ]:
(~gdf.is_valid).sum()

Можно посмотреть, какие именно объекты имеют некорректную геометрию:


In [ ]:
gdf[~gdf.is_valid]

Исправить некорректную геометрию можно так:


In [ ]:
gdf["geometry"] = gdf.make_valid()

После исправления стоит повторно проверить, остались ли объекты с некорректной геометрией:


In [ ]:
(~gdf.is_valid).sum()

## 3. Система координат (CRS)

Координаты геометрий в `GeoDataFrame` задаются в определённой **системе координат**
(**CRS — Coordinate Reference System**).

Перед выполнением пространственного анализа важно ответить на несколько вопросов:

- указана ли система координат в данных;
- какой у неё тип;
- подходит ли текущая CRS для расчёта расстояний и площадей;
- совпадает ли CRS у всех используемых слоёв.

В этом разделе мы рассмотрим основные характеристики CRS, которые можно определить на этапе первичного анализа.
Подробнее о системах координат мы поговорим во втором модуле.


### 3.1. Общая информация `.crs`

Атрибут `crs` позволяет определить, в какой системе координат записаны пространственные данные `GeoDataFrame`.

Если система координат не задана, значение атрибута `crs` будет равно `None`.


In [ ]:
gdf.crs

Здесь мы узнали, что система координат нашего набора данных имеет следующие характеристики:

- Это географическая система координат, ее идентификатор – EPSG:4326
- Название: WGS 84:
- Оси: используются координаты: широта (Lat) и долгота (Lon). Единицы измерения – градусы.
- Область применения данной системы координат: весь мир
- Используемый эллипсоид: WGS 84
- Нулевой меридиан: Гринвич


### 3.2. Тип системы координат

Система координат может быть:

- **географической** — координаты заданы в градусах (широта и долгота);
- **проекционной** — координаты заданы в линейных единицах (чаще всего в метрах).

Определить тип CRS можно с помощью следующих атрибутов: `.is_geographic` и `.is_projected`


In [ ]:
print("Географическая:", gdf.crs.is_geographic)
print("Проекционная:", gdf.crs.is_projected)

### 3.4. EPSG-код

Иногда бывает удобно получить **только числовой EPSG-код** системы координат.

Для этого используется метод `crs.to_epsg()`.


In [ ]:
gdf.crs.to_epsg()

Если CRS нестандартная, метод может вернуть `None`.


## 4. Атрибутивная информация

Помимо геометрии, каждый объект в `GeoDataFrame` содержит **атрибутивную информацию** — описательные характеристики, связанные с пространственными объектами.

В этом разделе мы рассмотрим, как анализировать структуру атрибутивных данных, проверять типы полей и выявлять пропущенные значения.


### 4.1. Список полей

Атрибут `columns` позволяет получить список всех полей в `DataFrame` или `GeoDataFrame`.
Результат представляет собой объект типа `Index`, содержащий названия всех полей, включая поле с геометрией.


In [ ]:
gdf.columns

### 4.2. Типы данных

Атрибут `dtypes` позволяет определить тип данных каждого поля в `DataFrame` или `GeoDataFrame`.


In [ ]:
gdf.dtypes

Типы данных необходимо проверять, чтобы убедиться, что каждое поле имеет корректный формат, с которым можно работать и выполнять необходимые преобразования без ошибок.


### 4.3. Пропуски в атрибутах

Для оценки качества атрибутивной информации важно проверить количество незаполненных значений по всем полям.


In [ ]:
gdf.isna().sum()

## 5. Другие методы изучения

Рассмотрим методы, которые не относятся к категориям выше.


### 5.1. Пространственный охват

**Bounding Box** — это минимальный прямоугольник, полностью охватывающий все объекты набора данных.

Он задаётся четырьмя координатами:

```
[minx, miny, maxx, maxy]
```

Его можно получить с помощью параметра `total_bounds`:


In [ ]:
gdf.total_bounds

## 6. Итог


В этом разделе мы познакомились с основными способами первичного анализа пространственного набора данных в формате `GeoDataFrame`.

Это очень важный этап проверки загруженных данных, так как позволяет понять структуру и содержание данных, убедиться в корректности загрузки, выявить потенциальные ошибки.
